In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import os
import json

SAVE_DIR    = "/content/drive/MyDrive/waste_project/"
IMG_SIZE    = (130, 130)   # same as Part 1
BATCH_SIZE  = 32
RANDOM_SEED = 42

# Load class names saved from Part 1
with open(SAVE_DIR + "class_names.json", 'r') as f:
    CLASS_NAMES = json.load(f)

NUM_CLASSES = len(CLASS_NAMES)

print("Classes:", CLASS_NAMES)
print("Num classes:", NUM_CLASSES)

# Load .npy files
print("\nLoading data...")
X_train = np.load(SAVE_DIR + "X_train.npy")
X_val   = np.load(SAVE_DIR + "X_val.npy")
X_test  = np.load(SAVE_DIR + "X_test.npy")
y_train = np.load(SAVE_DIR + "y_train.npy")
y_val   = np.load(SAVE_DIR + "y_val.npy")
y_test  = np.load(SAVE_DIR + "y_test.npy")

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")

# Load class weights saved from Part 1
with open(SAVE_DIR + "class_weights.json", 'r') as f:
    cw_raw = json.load(f)
CLASS_WEIGHTS = {int(k): float(v) for k, v in cw_raw.items()}

print("\nClass weights:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:12s}: {CLASS_WEIGHTS[i]:.3f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash', 'biological', 'battery', 'shoes', 'clothes']
Num classes: 10

Loading data...
X_train : (10860, 130, 130, 3)
X_val   : (3103, 130, 130, 3)
X_test  : (1552, 130, 130, 3)

Class weights:
  cardboard   : 1.740
  glass       : 0.771
  metal       : 2.019
  paper       : 1.478
  plastic     : 1.792
  trash       : 2.225
  biological  : 1.576
  battery     : 1.640
  shoes       : 0.785
  clothes     : 0.291


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D,
    BatchNormalization, Input
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

# Confirm GPU is available before training
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU available: {gpus[0].name}")
else:
    print("\n⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU")

TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

✅ GPU available: /physical_device:GPU:0


In [ ]:
# One-hot encode labels
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

# Augmentation for training
# preprocess_input is critical — converts [0,255] → [-1,+1]
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2]
)

# No augmentation for val/test — only preprocessing
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow(
    X_train, y_train_cat,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_gen = val_datagen.flow(
    X_val, y_val_cat,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Generators ready!")
print(f"  Train batches/epoch : {len(X_train) // BATCH_SIZE}")
print(f"  Val   batches/epoch : {len(X_val)   // BATCH_SIZE}")
print(f"  preprocess_input    : ✅  [-1, +1] scale")

Generators ready!
  Train batches/epoch : 339
  Val   batches/epoch : 96
  preprocess_input    : ✅  [-1, +1] scale


In [ ]:
def build_model(num_classes, learning_rate=0.001):
    """
    MobileNetV2 pretrained on ImageNet as base.
    Custom classification head on top.
    Input: 160x160x3 (matches IMG_SIZE from Part 1)
    """
    # Load MobileNetV2 without its top classification layer
    base_model = MobileNetV2(
        input_shape=(130, 130, 3),
        include_top=False,
        weights='imagenet'
    )

    # Freeze all base layers for Stage 1
    base_model.trainable = False

    print(f"Base model layers     : {len(base_model.layers)}")
    print(f"Trainable layers now  : 0  (all frozen for Stage 1)")

    # Build classification head
    inputs = Input(shape=(130, 130, 3))

    x = base_model(inputs, training=False)
    # training=False keeps BatchNorm in inference mode while frozen

    x = GlobalAveragePooling2D()(x)
    # (5,5,1280) → (1280,)

    x = BatchNormalization()(x)

    x = Dense(512, activation='relu')(x)
    x = Dropout(0.4)(x)

    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)

    outputs = Dense(num_classes, activation='softmax')(x)
    # 10 probabilities that sum to 1.0

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model, base_model


model, base_model = build_model(NUM_CLASSES, learning_rate=0.001)
model.summary()

/tmp/ipykernel_4320/453007804.py:8: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Base model layers     : 154
Trainable layers now  : 0  (all frozen for Stage 1)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 130, 130, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,052,874 (11.65 MB)

 Trainable params: 792,330 (3.02 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

In [ ]:
MODEL_PATH_S1 = SAVE_DIR + "waste_model_stage1.keras"

callbacks_stage1 = [

    ModelCheckpoint(
        filepath=MODEL_PATH_S1,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor='val_accuracy',
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("Stage 1 callbacks ready!")
print(f"Best model saved to: {MODEL_PATH_S1}")

Stage 1 callbacks ready!
Best model saved to: /content/drive/MyDrive/waste_project/waste_model_stage1.keras


In [ ]:
print("=" * 55)
print("  STAGE 1 — Training head only (base frozen)")
print("=" * 55)

EPOCHS_S1 = 25

history_s1 = model.fit(
    train_gen,
    steps_per_epoch  = len(X_train) // BATCH_SIZE,
    validation_data  = val_gen,
    validation_steps = len(X_val) // BATCH_SIZE,
    epochs           = EPOCHS_S1,
    callbacks        = callbacks_stage1,
    class_weight     = CLASS_WEIGHTS,
    verbose          = 1
)

val_loss_s1, val_acc_s1 = model.evaluate(
    val_gen,
    steps=len(X_val) // BATCH_SIZE,
    verbose=0
)

print(f"\nStage 1 complete!")
print(f"  Best val accuracy : {max(history_s1.history['val_accuracy'])*100:.2f}%")
print(f"  Final val accuracy: {val_acc_s1*100:.2f}%")
print(f"  Final val loss    : {val_loss_s1:.4f}")

  STAGE 1 — Training head only (base frozen)
Epoch 1/25
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step - accuracy: 0.7045 - loss: 1.0959
Epoch 1: val_accuracy improved from None to 0.89583, saving model to /content/drive/MyDrive/waste_project/waste_model_stage1.keras

Epoch 1: finished saving model to /content/drive/MyDrive/waste_project/waste_model_stage1.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 101s 228ms/step - accuracy: 0.7740 - loss: 0.8219 - val_accuracy: 0.8958 - val_loss: 0.3172 - learning_rate: 0.0010
Epoch 2/25
  1/339 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8438 - loss: 0.6107

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 2: val_accuracy improved from 0.89583 to 0.89779, saving model to /content/drive/MyDrive/waste_project/waste_model_stage1.keras

Epoch 2: finished saving model to /content/drive/MyDrive/waste_project/waste_model_stage1.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8438 - loss: 0.6107 - val_accuracy: 0.8978 - val_loss: 0.3151 - learning_rate: 0.0010
Epoch 3/25
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 0.8440 - loss: 0.5428
Epoch 3: val_accuracy improved from 0.89779 to 0.91113, saving model to /content/drive/MyDrive/waste_project/waste_model_stage1.keras

Epoch 3: finished saving model to /content/drive/MyDrive/waste_project/waste_model_stage1.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 104s 172ms/step - accuracy: 0.8447 - loss: 0.5254 - val_accuracy: 0.9111 - val_loss: 0.2590 - learning_rate: 0.0010
Epoch 4/25
  1/339 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9062 - loss: 0.2970
Epoch 4: val_accuracy did not improve from 0.91113
339/339 ━━━━━━━━━━━━━━━━━

In [ ]:
def plot_history(history, stage_name):
    acc      = history.history['accuracy']
    val_acc  = history.history['val_accuracy']
    loss     = history.history['loss']
    val_loss = history.history['val_loss']
    epochs   = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Training Results — {stage_name}', fontsize=14)

    ax1.plot(epochs, acc,     'b-o', markersize=4, label='Train accuracy')
    ax1.plot(epochs, val_acc, 'r-o', markersize=4, label='Val accuracy')
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0, 1])

    ax2.plot(epochs, loss,     'b-o', markersize=4, label='Train loss')
    ax2.plot(epochs, val_loss, 'r-o', markersize=4, label='Val loss')
    ax2.set_title('Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = stage_name.lower().replace(' ', '_').replace('—','').strip()
    save_path = SAVE_DIR + f"plot_{fname}.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Plot saved: {save_path}")


plot_history(history_s1, "Stage 1 — Transfer Learning")

Plot saved: /content/drive/MyDrive/waste_project/plot_stage_1__transfer_learning.png


In [ ]:
best_s1 = max(history_s1.history['val_accuracy'])

print("=" * 50)
print(f"  Stage 1 best val accuracy: {best_s1*100:.2f}%")
print("=" * 50)

if best_s1 >= 0.85:
    print("\n✅ Accuracy is GOOD (≥85%) — fine-tuning will push it further")
elif best_s1 >= 0.75:
    print("\n⚠️  Accuracy is ACCEPTABLE (75–85%) — fine-tuning RECOMMENDED")
else:
    print("\n❌ Accuracy is LOW (<75%) — fine-tuning REQUIRED")

print("\nRunning Stage 2 fine-tuning regardless for best results...")

  Stage 1 best val accuracy: 94.47%

✅ Accuracy is GOOD (≥85%) — fine-tuning will push it further

Running Stage 2 fine-tuning regardless for best results...


In [ ]:
print("=" * 55)
print("  STAGE 2 — Fine-tuning last 40 layers of MobileNetV2")
print("=" * 55)

# Load best weights from Stage 1
model = tf.keras.models.load_model(MODEL_PATH_S1)
print("Loaded best Stage 1 model")

# Find the MobileNetV2 base inside the model
base_model = None
for layer in model.layers:
    if 'mobilenetv2' in layer.name.lower():
        base_model = layer
        break

if base_model is None:
    # Fallback: rebuild and reload weights
    model, base_model = build_model(NUM_CLASSES)
    model.load_weights(MODEL_PATH_S1)
    print("Rebuilt model and loaded weights")

# Unfreeze the entire base first
base_model.trainable = True

# Re-freeze everything EXCEPT the last 40 layers
FINE_TUNE_FROM = len(base_model.layers) - 40

for layer in base_model.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

for layer in base_model.layers[FINE_TUNE_FROM:]:
    layer.trainable = True

trainable = sum(1 for l in model.layers if l.trainable)
print(f"\nTotal model layers      : {len(model.layers)}")
print(f"MobileNetV2 layers      : {len(base_model.layers)}")
print(f"Fine-tuning from layer  : {FINE_TUNE_FROM} onwards")
print(f"Trainable layers now    : {trainable}")

# CRITICAL: use very small learning rate for fine-tuning
# Large LR would destroy the pretrained weights
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✅ Fine-tuning compiled with lr=1e-5")

  STAGE 2 — Fine-tuning last 40 layers of MobileNetV2
Loaded best Stage 1 model

Total model layers      : 9
MobileNetV2 layers      : 154
Fine-tuning from layer  : 114 onwards
Trainable layers now    : 9

✅ Fine-tuning compiled with lr=1e-5


In [ ]:
MODEL_PATH_S2 = SAVE_DIR + "waste_model_stage2.keras"

callbacks_stage2 = [

    ModelCheckpoint(
        filepath=MODEL_PATH_S2,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor='val_accuracy',
        patience=8,              # more patience for fine-tuning
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-8,
        verbose=1
    )
]

print("Stage 2 callbacks ready!")
print(f"Best model saved to: {MODEL_PATH_S2}")

Stage 2 callbacks ready!
Best model saved to: /content/drive/MyDrive/waste_project/waste_model_stage2.keras


In [ ]:
print("Starting fine-tuning...\n")

EPOCHS_S2 = 10

history_s2 = model.fit(
    train_gen,
    steps_per_epoch  = len(X_train) // BATCH_SIZE,
    validation_data  = val_gen,
    validation_steps = len(X_val) // BATCH_SIZE,
    epochs           = EPOCHS_S2,
    callbacks        = callbacks_stage2,
    class_weight     = CLASS_WEIGHTS,
    verbose          = 1
)

val_loss_s2, val_acc_s2 = model.evaluate(
    val_gen,
    steps=len(X_val) // BATCH_SIZE,
    verbose=0
)

best_s2 = max(history_s2.history['val_accuracy'])

print(f"\nStage 2 complete!")
print(f"  Best val accuracy    : {best_s2*100:.2f}%")
print(f"  Stage 1 best was     : {best_s1*100:.2f}%")
print(f"  Improvement          : +{(best_s2 - best_s1)*100:.2f}%")

Starting fine-tuning...

Epoch 1/30
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.8239 - loss: 0.5855
Epoch 1: val_accuracy improved from None to 0.93490, saving model to /content/drive/MyDrive/waste_project/waste_model_stage2.keras

Epoch 1: finished saving model to /content/drive/MyDrive/waste_project/waste_model_stage2.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 100s 224ms/step - accuracy: 0.8364 - loss: 0.5249 - val_accuracy: 0.9349 - val_loss: 0.2307 - learning_rate: 1.0000e-05
Epoch 2/30
  1/339 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.8438 - loss: 0.5635
Epoch 2: val_accuracy did not improve from 0.93490
339/339 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8438 - loss: 0.5635 - val_accuracy: 0.9349 - val_loss: 0.2307 - learning_rate: 1.0000e-05
Epoch 3/30
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step - accuracy: 0.8716 - loss: 0.4148
Epoch 3: val_accuracy improved from 0.93490 to 0.93555, saving model to /content/drive/MyDrive/waste_project/waste_model_stage2.keras

Epoc

In [ ]:
plot_history(history_s2, "Stage 2 — Fine-tuning")

print("\n" + "="*45)
print("  ACCURACY COMPARISON")
print("="*45)
print(f"  Stage 1 (transfer) : {best_s1*100:.2f}%")
print(f"  Stage 2 (fine-tune): {best_s2*100:.2f}%")
print("="*45)

Plot saved: /content/drive/MyDrive/waste_project/plot_stage_2__fine-tuning.png

  ACCURACY COMPARISON
  Stage 1 (transfer) : 94.47%
  Stage 2 (fine-tune): 93.59%


In [ ]:
# Pick the better stage
if best_s2 >= best_s1:
    print(f"Stage 2 is better ({best_s2*100:.2f}%) — using Stage 2 weights")
    best_model = tf.keras.models.load_model(MODEL_PATH_S2)
else:
    print(f"Stage 1 is better ({best_s1*100:.2f}%) — using Stage 1 weights")
    best_model = tf.keras.models.load_model(MODEL_PATH_S1)

# Save weights — avoids ALL Keras version mismatch errors on PC
WEIGHTS_PATH = SAVE_DIR + "waste_weights_FINAL.weights.h5"
best_model.save_weights(WEIGHTS_PATH)

size_mb = os.path.getsize(WEIGHTS_PATH) / (1024 * 1024)
print(f"\n✅ Saved: waste_weights_FINAL.weights.h5")
print(f"   Size : {size_mb:.1f} MB")
print(f"   This file goes in your project folder on your PC")

Stage 1 is better (94.47%) — using Stage 1 weights

✅ Saved: waste_weights_FINAL.weights.h5
   Size : 18.1 MB
   This file goes in your project folder on your PC


In [ ]:
# Test one image from test set before full evaluation
sample_img        = X_test[0]
sample_true_label = int(y_test[0])

# Preprocess the same way as training
import numpy as np
sample_processed = preprocess_input(sample_img.copy())
sample_batch     = np.expand_dims(sample_processed, axis=0)

pred_probs    = best_model.predict(sample_batch, verbose=0)[0]
predicted_idx = int(np.argmax(pred_probs))
confidence    = float(pred_probs[predicted_idx]) * 100

print("Sanity check on one test image:\n")
print(f"  True label      : {CLASS_NAMES[sample_true_label]}")
print(f"  Predicted label : {CLASS_NAMES[predicted_idx]}")
print(f"  Confidence      : {confidence:.1f}%")
print(f"  Correct?        : {'✅ YES' if predicted_idx == sample_true_label else '❌ NO'}")

print("\nAll class probabilities:")
for i, name in enumerate(CLASS_NAMES):
    bar = "█" * int(pred_probs[i] * 30)
    print(f"  {name:12s}: {pred_probs[i]*100:5.1f}%  {bar}")

Sanity check on one test image:

  True label      : glass
  Predicted label : plastic
  Confidence      : 81.1%
  Correct?        : ❌ NO

All class probabilities:
  cardboard   :   0.0%  
  glass       :   6.9%  ██
  metal       :  11.7%  ███
  paper       :   0.2%  
  plastic     :  81.1%  ████████████████████████
  trash       :   0.0%  
  biological  :   0.0%  
  battery     :   0.0%  
  shoes       :   0.0%  
  clothes     :   0.0%  


In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
import seaborn as sns
import pandas as pd

print("Running predictions on entire test set...\n")

# Preprocess all test images
X_test_processed = preprocess_input(X_test.copy())

# Predict in one batch
all_probs  = best_model.predict(X_test_processed, batch_size=32, verbose=1)
y_pred     = np.argmax(all_probs, axis=1)
y_true     = y_test.astype(int)
confidences = all_probs.max(axis=1) * 100

overall_acc = accuracy_score(y_true, y_pred)

print(f"\nOverall test accuracy : {overall_acc*100:.2f}%")
print(f"Correct predictions   : {(y_pred == y_true).sum()} / {len(y_true)}")
print(f"Wrong predictions     : {(y_pred != y_true).sum()}")
print(f"Average confidence    : {confidences.mean():.1f}%")

Running predictions on entire test set...

49/49 ━━━━━━━━━━━━━━━━━━━━ 16s 247ms/step

Overall test accuracy : 93.36%
Correct predictions   : 1449 / 1552
Wrong predictions     : 103
Average confidence    : 94.6%


In [ ]:
report = classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    digits=3
)

print("=" * 60)
print("  CLASSIFICATION REPORT — Per Class Breakdown")
print("=" * 60)
print(report)

  CLASSIFICATION REPORT — Per Class Breakdown
              precision    recall  f1-score   support

   cardboard      0.952     0.899     0.925        89
       glass      0.923     0.891     0.906       201
       metal      0.763     0.922     0.835        77
       paper      0.931     0.895     0.913       105
     plastic      0.772     0.826     0.798        86
       trash      0.865     0.914     0.889        70
  biological      0.960     0.960     0.960        99
     battery      0.937     0.947     0.942        94
       shoes      0.959     0.949     0.954       198
     clothes      0.989     0.972     0.980       533

    accuracy                          0.934      1552
   macro avg      0.905     0.917     0.910      1552
weighted avg      0.937     0.934     0.935      1552



In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'shrink': 0.8}
)

ax.set_title('Confusion Matrix — Test Set', fontsize=14, pad=15)
ax.set_xlabel('Predicted Label', fontsize=12, labelpad=10)
ax.set_ylabel('True Label', fontsize=12, labelpad=10)
ax.tick_params(axis='x', rotation=35)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(SAVE_DIR + "confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved!")

Confusion matrix saved!


In [ ]:
print("Per-class accuracy breakdown:\n")
for i, name in enumerate(CLASS_NAMES):
    total   = cm[i].sum()
    correct = cm[i, i]
    acc     = correct / total * 100 if total > 0 else 0

    # Most confused with
    row        = cm[i].copy()
    row[i]     = 0
    confused_i = row.argmax()
    confused_n = row[confused_i]

    bar = "█" * int(acc / 5)
    print(f"  {name:12s}: {acc:5.1f}%  {bar}", end="")
    if confused_n > 0:
        print(f"  | confused with '{CLASS_NAMES[confused_i]}' ({confused_n}x)")
    else:
        print("  | no confusion!")

Per-class accuracy breakdown:

  cardboard   :  89.9%  █████████████████  | confused with 'paper' (3x)
  glass       :  89.1%  █████████████████  | confused with 'plastic' (11x)
  metal       :  92.2%  ██████████████████  | confused with 'plastic' (3x)
  paper       :  89.5%  █████████████████  | confused with 'plastic' (4x)
  plastic     :  82.6%  ████████████████  | confused with 'glass' (9x)
  trash       :  91.4%  ██████████████████  | confused with 'glass' (2x)
  biological  :  96.0%  ███████████████████  | confused with 'metal' (2x)
  battery     :  94.7%  ██████████████████  | confused with 'metal' (4x)
  shoes       :  94.9%  ██████████████████  | confused with 'plastic' (2x)
  clothes     :  97.2%  ███████████████████  | confused with 'trash' (6x)


In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

colors = [
    '#4e79a7','#59a14f','#f28e2b','#e15759','#76b7b2',
    '#b07aa1','#ff9da7','#9c755f','#bab0ac','#edc948'
]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(CLASS_NAMES, per_class_acc,
              color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, per_class_acc):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val:.1f}%', ha='center', va='bottom',
        fontsize=10, fontweight='bold'
    )

ax.axhline(
    y=overall_acc * 100,
    color='red', linestyle='--', linewidth=1.5,
    label=f'Overall: {overall_acc*100:.1f}%'
)

ax.set_title('Per-Class Accuracy on Test Set', fontsize=14, pad=12)
ax.set_xlabel('Waste Category', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_ylim(0, 115)
ax.tick_params(axis='x', rotation=25)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR + "per_class_accuracy.png", dpi=150)
plt.show()
print("Per-class chart saved!")

Per-class chart saved!


In [ ]:
correct_mask = (y_pred == y_true)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: correct vs wrong confidence histogram
axes[0].hist(confidences[correct_mask],  bins=25, alpha=0.7,
             color='#59a14f', label=f'Correct ({correct_mask.sum()})')
axes[0].hist(confidences[~correct_mask], bins=25, alpha=0.7,
             color='#e15759', label=f'Wrong ({(~correct_mask).sum()})')
axes[0].set_title('Confidence Distribution', fontsize=12)
axes[0].set_xlabel('Confidence (%)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: avg confidence per class
avg_conf = []
for i in range(NUM_CLASSES):
    mask = (y_pred == i)
    avg_conf.append(confidences[mask].mean() if mask.sum() > 0 else 0)

axes[1].bar(CLASS_NAMES, avg_conf, color=colors, edgecolor='white')
axes[1].set_title('Avg Confidence per Predicted Class', fontsize=12)
axes[1].set_xlabel('Predicted Class')
axes[1].set_ylabel('Avg Confidence (%)')
axes[1].set_ylim(0, 115)
axes[1].tick_params(axis='x', rotation=25)
axes[1].grid(axis='y', alpha=0.3)

for i, (bar, val) in enumerate(zip(axes[1].patches, avg_conf)):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{val:.0f}%', ha='center', va='bottom', fontsize=9
    )

plt.tight_layout()
plt.savefig(SAVE_DIR + "confidence_distribution.png", dpi=150)
plt.show()
print("Confidence chart saved!")

Confidence chart saved!


In [ ]:
wrong_indices = np.where(y_pred != y_true)[0]
print(f"Total wrong predictions: {len(wrong_indices)}")

n_show = min(12, len(wrong_indices))
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle(f'Wrong Predictions (showing {n_show} of {len(wrong_indices)})',
             fontsize=13)

for idx, ax in zip(wrong_indices[:n_show], axes.flat):
    img        = X_test[idx].astype('uint8')
    true_lbl   = CLASS_NAMES[y_true[idx]]
    pred_lbl   = CLASS_NAMES[y_pred[idx]]
    conf       = all_probs[idx][y_pred[idx]] * 100

    ax.imshow(img)
    ax.axis('off')
    ax.set_title(
        f'True: {true_lbl}\nPred: {pred_lbl} ({conf:.0f}%)',
        fontsize=8, color='red'
    )

for ax in axes.flat[n_show:]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(SAVE_DIR + "wrong_predictions.png", dpi=120, bbox_inches='tight')
plt.show()
print("Wrong predictions chart saved!")

Total wrong predictions: 103
Wrong predictions chart saved!


In [ ]:
print("\n" + "=" * 58)
print("  FINAL SUMMARY")
print("=" * 58)
print(f"  Model              : MobileNetV2 + Custom Head")
print(f"  Classes            : {NUM_CLASSES}")
print(f"  Class names        : {CLASS_NAMES}")
print(f"  Image size         : {IMG_SIZE}")
print(f"  Test set size      : {len(y_true)} images")
print(f"  Overall accuracy   : {overall_acc*100:.2f}%")
print(f"  Avg confidence     : {confidences.mean():.1f}%")
print(f"  Stage 1 best       : {best_s1*100:.2f}%")
print(f"  Stage 2 best       : {best_s2*100:.2f}%")
print("-" * 58)
print("  Per-class accuracy:")
for i, name in enumerate(CLASS_NAMES):
    total   = cm[i].sum()
    correct = cm[i, i]
    acc     = correct / total * 100 if total > 0 else 0
    bar     = "█" * int(acc / 5)
    print(f"    {name:12s}: {acc:5.1f}%  {bar}")
print("-" * 58)
print("  Files saved to Google Drive:")
for fname in [
    "waste_weights_FINAL.weights.h5",
    "confusion_matrix.png",
    "per_class_accuracy.png",
    "confidence_distribution.png",
    "wrong_predictions.png",
    "plot_stage_1_transfer_learning.png",
    "plot_stage_2_fine_tuning.png",
]:
    fpath = SAVE_DIR + fname
    if os.path.exists(fpath):
        size  = os.path.getsize(fpath) / (1024 * 1024)
        print(f"    ✅ {fname}  ({size:.1f} MB)")
    else:
        print(f"    ⚠️  {fname}  (not found)")
print("=" * 58)

# Download the weights file to your PC
from google.colab import files
print("\nDownloading waste_weights_FINAL.weights.h5...")
files.download(SAVE_DIR + "waste_weights_FINAL.weights.h5")
print("✅ Download started — save this file in your project folder")


  FINAL SUMMARY
  Model              : MobileNetV2 + Custom Head
  Classes            : 10
  Class names        : ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash', 'biological', 'battery', 'shoes', 'clothes']
  Image size         : (130, 130)
  Test set size      : 1552 images
  Overall accuracy   : 93.36%
  Avg confidence     : 94.6%
  Stage 1 best       : 94.47%
  Stage 2 best       : 93.59%
----------------------------------------------------------
  Per-class accuracy:
    cardboard   :  89.9%  █████████████████
    glass       :  89.1%  █████████████████
    metal       :  92.2%  ██████████████████
    paper       :  89.5%  █████████████████
    plastic     :  82.6%  ████████████████
    trash       :  91.4%  ██████████████████
    biological  :  96.0%  ███████████████████
    battery     :  94.7%  ██████████████████
    shoes       :  94.9%  ██████████████████
    clothes     :  97.2%  ███████████████████
----------------------------------------------------------
  Fi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started — save this file in your project folder
